In [3]:
"""
Experiments 6 & 7: Predictive uncertainty and calibration analysis.

Standalone script. Loads Nordpool data fresh for each price area, builds
phase-space-reconstructed train/test sets per horizon, trains
Gaussian SDE-Net (alpha=2.0) and LevySDE-Net (alpha=1.5) per area x
horizon x seed, and computes:
  - Experiment 6: ensemble spread, ensemble variance, 95% interval width
  - Experiment 7: empirical coverage of nominal 50/80/95% intervals

Uses the SAME model architecture and training procedure as the rest of
the project's notebook (SDENetSingle: SGD dual-optimizer, MSE readout
loss + adversarial BCE diffusion loss) -- NOT a physics-loss scheme.

Does not depend on any prior notebook state. Run top to bottom.
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import levy_stable
from sklearn.preprocessing import StandardScaler

# ============================================================
# CONFIG
# ============================================================
DATA_PATH = "NordpoolMarket.csv"
TARGET_COLUMN = "SpotPriceEUR"

# TODO: confirm these PriceArea string values match your CSV exactly
# (e.g. check whether NO3/NO4 are separate rows or need combining).
AREAS = ["DK1"]

HORIZONS = [1, 2, 4]          # forecast steps ahead (hours) -- matches Exp 6/7 table columns
TAU = 1
EMBED_DIM = 8                 # matches m=8 used elsewhere for DK1; NOT re-validated per area
TEST_FRACTION = 0.2

SEEDS = [0, 1, 2, 3, 4]
N_ENSEMBLE = 200               # TODO: confirm/justify this choice before reporting in the paper
EPOCHS = 100
ITERS = 50
LAYER_DEPTH = 6

ALPHA_CONFIGS = {
    "Gaussian SDE-Net": 2.0,
    "LévySDE-Net": 1.5,
}

CI_LEVELS = [0.50, 0.80, 0.95]  # for Experiment 7 coverage

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Data loading (parameterized by area; mirrors the DK1-only cell
# in the notebook, generalized)
# ============================================================
def load_area_price_diffs(area):
    data = pd.read_csv(DATA_PATH, sep=";", decimal=",")
    data = data[data["PriceArea"] == area].copy()
    data = data[["HourUTC", TARGET_COLUMN]].copy()
    data.columns = ["date", "close"]
    data["close"] = pd.to_numeric(data["close"], errors="coerce")
    data = data.dropna(subset=["close"]).reset_index(drop=True)
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data = data.dropna(subset=["date"])
    data = data.sort_values("date").drop_duplicates(subset="date", keep="last").reset_index(drop=True)

    if data.empty:
        raise ValueError(f"No rows found for PriceArea == '{area}' -- check the area string against the CSV.")

    prices = data["close"].astype(float).values
    dates = data["date"].values
    diffs = np.diff(prices)
    diff_dates = dates[1:]  # diffs[i] corresponds to dates[i+1]

    n = len(diffs)
    split_idx = int(n * (1 - TEST_FRACTION))

    train_diffs = diffs[:split_idx].reshape(-1, 1)
    test_diffs = diffs[split_idx:].reshape(-1, 1)
    train_dates = diff_dates[:split_idx]
    test_dates = diff_dates[split_idx:]

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_diffs).flatten()
    test_scaled = scaler.transform(test_diffs).flatten()

    train_df = pd.DataFrame({"Date": train_dates, "closescale": train_scaled})
    test_df = pd.DataFrame({"Date": test_dates, "closescale": test_scaled})
    return train_df, test_df, scaler


# ============================================================
# Phase-space reconstruction (direct copy of the notebook's PhaSpaRecon)
# ============================================================
def PhaSpaRecon(df, tau, d, T):
    data = np.array(df)[:, 1]
    lens = len(data)
    M = lens - (d - 1) * tau - T  # usable (X, Y) pairs for this horizon
    if M < 1:
        raise ValueError(
            f"delay time, embedding dimension, or horizon too large for this series "
            f"length (lens={lens}, tau={tau}, d={d}, T={T})"
        )
    Xn1 = np.zeros((d, M))
    for i in range(d):
        Xn1[i, :] = data[i * tau: i * tau + M]
    start_y = (d - 1) * tau + T
    Yn1 = data[start_y: start_y + M]
    return Xn1.T, Yn1


def build_horizon_data(train_df, test_df, tau, d, horizon):
    x_train, y_train = PhaSpaRecon(train_df, tau=tau, d=d, T=horizon)
    x_test, y_test = PhaSpaRecon(test_df, tau=tau, d=d, T=horizon)
    return (
        np.asarray(x_train, dtype=np.float64),
        np.asarray(y_train, dtype=np.float64),
        np.asarray(x_test, dtype=np.float64),
        np.asarray(y_test, dtype=np.float64),
    )


# ============================================================
# Model: exact architecture from the notebook (cell 63's SDENetSingle)
# ============================================================
class Drift(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.fc = nn.Linear(seq_len, seq_len)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, t, x):
        return self.relu(self.fc(x))


class Diffusion(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.fc1 = nn.Linear(seq_len, 100)
        self.fc2 = nn.Linear(100, 1)

    def forward(self, t, x):
        out = self.relu(self.fc1(x))
        out = self.fc2(out)
        return torch.sigmoid(out)


class SDENetSingle(nn.Module):
    def __init__(self, layer_depth, alpha, seq_len, beta=0.0, levy_scale=0.1):
        super().__init__()
        self.layer_depth = layer_depth
        self.alpha = alpha
        self.beta = beta
        self.levy_scale = levy_scale
        self.seq_len = seq_len
        self.downsampling_layers = nn.Linear(seq_len, seq_len)
        self.drift = Drift(seq_len)
        self.diffusion = Diffusion(seq_len)
        self.deltat = 1.0 / self.layer_depth
        self.sigma = 0.5

    def forward(self, x, training_diffusion=False):
        out = self.downsampling_layers(x)
        if not training_diffusion:
            t = 0
            diffusion_term = self.sigma * self.diffusion(t, out)
            for i in range(self.layer_depth):
                t = float(i) / self.layer_depth
                noise = torch.from_numpy(
                    levy_stable.rvs(self.alpha, self.beta, size=(x.shape[0], self.seq_len), scale=self.levy_scale)
                ).to(x)
                out = out + self.drift(t, out) * self.deltat \
                    + diffusion_term * (self.deltat ** (1 / self.alpha)) * noise
            Drift_out = self.drift(t, out)
            return Drift_out, out
        else:
            return self.diffusion(0, out.detach())


def train_sdenet(x_train, y_train, alpha, seed, epochs=EPOCHS, iters=ITERS, layer_depth=LAYER_DEPTH):
    """Trains one SDE-Net; identical training scheme to the notebook's
    train_sdenet_predict, but returns the trained model itself (not a
    single-shot prediction) so we can draw an ensemble at eval time."""
    set_seed(seed)
    seq_len = x_train.shape[1]
    model = SDENetSingle(layer_depth=layer_depth, alpha=alpha, seq_len=seq_len).to(device)

    optimizer_F = optim.SGD(
        [{"params": model.downsampling_layers.parameters()}, {"params": model.drift.parameters()}],
        lr=1e-4, momentum=0.9, weight_decay=5e-4,
    )
    optimizer_G = optim.SGD(
        [{"params": model.diffusion.parameters()}],
        lr=1e-2, momentum=0.9, weight_decay=5e-4,
    )
    criterion_bce = nn.BCELoss()

    Xtr = torch.from_numpy(x_train).float().to(device)
    ytr = torch.from_numpy(y_train).float().view(-1, 1).to(device)
    real_label, fake_label = 0, 1
    batch_size = Xtr.shape[0]

    model.train()
    for _ in range(epochs):
        for _ in range(iters):
            optimizer_F.zero_grad()
            drift_out, out = model(Xtr)
            pred = out.mean(dim=1, keepdim=True)
            loss = torch.mean((ytr - pred) ** 2)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(model.downsampling_layers.parameters()) + list(model.drift.parameters()),
                max_norm=5.0,
            )
            optimizer_F.step()

            label_real = torch.full((batch_size, 1), real_label, device=device, dtype=torch.float32)
            optimizer_G.zero_grad()
            predict_in = model(Xtr, training_diffusion=True)
            loss_in = criterion_bce(predict_in, label_real)
            loss_in.backward()

            label_fake = torch.full((batch_size, 1), fake_label, device=device, dtype=torch.float32)
            inputs_out = 2 * torch.randn(batch_size, seq_len, device=device) + Xtr
            predict_out = model(inputs_out, training_diffusion=True)
            loss_out = criterion_bce(predict_out, label_fake)
            loss_out.backward()
            optimizer_G.step()

    model.eval()
    return model


def ensemble_predict(model, x_test, n_ensemble):
    """Draw n_ensemble stochastic forward passes from a trained model
    on the (fixed) test inputs. Returns array of shape (n_ensemble, n_test_points)."""
    Xte = torch.from_numpy(x_test).float().to(device)
    samples = np.zeros((n_ensemble, x_test.shape[0]))
    with torch.no_grad():
        for k in range(n_ensemble):
            _, out = model(Xte)
            samples[k, :] = out.mean(dim=1).cpu().numpy()
    return samples


def ensemble_stats(samples, ci_level=0.95):
    """samples: (n_ensemble, n_test_points)."""
    spread = samples.std(axis=0)       # per-point ensemble spread (std)
    variance = samples.var(axis=0)
    lo_pct = (1 - ci_level) / 2 * 100
    hi_pct = (1 + ci_level) / 2 * 100
    lower = np.percentile(samples, lo_pct, axis=0)
    upper = np.percentile(samples, hi_pct, axis=0)
    width = upper - lower
    return spread, variance, width, lower, upper


def empirical_coverage(y_true, lower, upper):
    inside = (y_true >= lower) & (y_true <= upper)
    return inside.mean()


# ============================================================
# Main loop
# ============================================================
uncertainty_rows = []
calibration_rows = []

for area in AREAS:
    print(f"\n{'='*60}\nAREA: {area}\n{'='*60}")
    try:
        train_df, test_df, scaler = load_area_price_diffs(area)
    except ValueError as e:
        print(f"  SKIPPING {area}: {e}")
        continue

    for model_name, alpha in ALPHA_CONFIGS.items():
        print(f"\n  Model: {model_name} (alpha={alpha})")

        # Per-horizon storage across seeds, for averaging
        spread_by_horizon = {q: [] for q in HORIZONS}
        width95_by_horizon = {q: [] for q in HORIZONS}
        coverage_by_horizon = {q: {ci: [] for ci in CI_LEVELS} for q in HORIZONS}

        for q in HORIZONS:
            x_train, y_train, x_test, y_test = build_horizon_data(
                train_df, test_df, tau=TAU, d=EMBED_DIM, horizon=q
            )

            for seed in SEEDS:
                print(f"    horizon q={q}, seed={seed} ...")
                model = train_sdenet(x_train, y_train, alpha=alpha, seed=seed)
                samples = ensemble_predict(model, x_test, N_ENSEMBLE)

                spread, variance, width95, lower95, upper95 = ensemble_stats(samples, ci_level=0.95)
                spread_by_horizon[q].append(spread.mean())
                width95_by_horizon[q].append(width95.mean())
                coverage_by_horizon[q][0.95].append(empirical_coverage(y_test, lower95, upper95))

                for ci in [c for c in CI_LEVELS if c != 0.95]:
                    _, _, _, lower_c, upper_c = ensemble_stats(samples, ci_level=ci)
                    coverage_by_horizon[q][ci].append(empirical_coverage(y_test, lower_c, upper_c))

        # ---- Experiment 6 row: avg ensemble spread (q=1) + width at q=1,2,4 ----
        row6 = {
            "Area": area,
            "Model": model_name,
            "Avg_Ensemble_Spread_q1": np.mean(spread_by_horizon[HORIZONS[0]]),
        }
        for q in HORIZONS:
            row6[f"Width_q{q}_mean"] = np.mean(width95_by_horizon[q])
            row6[f"Width_q{q}_std"] = np.std(width95_by_horizon[q], ddof=1)
        uncertainty_rows.append(row6)

        # ---- Experiment 7 row: coverage at 50/80/95%, reported at q=1 ----
        # TODO: decide whether calibration should be reported per-horizon
        # or aggregated (here: q=1, matching the table's implicit single
        # coverage triple per model/area).
        row7 = {"Area": area, "Model": model_name}
        for ci in CI_LEVELS:
            row7[f"Coverage_{int(ci*100)}pct"] = np.mean(coverage_by_horizon[HORIZONS[0]][ci])
        calibration_rows.append(row7)

# ============================================================
# Save and print paper-style tables
# ============================================================
uncertainty_df = pd.DataFrame(uncertainty_rows)
calibration_df = pd.DataFrame(calibration_rows)

uncertainty_df.to_csv("experiment6_uncertainty_spread_by_area.csv", index=False)
calibration_df.to_csv("experiment7_calibration_by_area.csv", index=False)

print("\n\n===== EXPERIMENT 6: PREDICTIVE UNCERTAINTY =====")
print(uncertainty_df.to_string(index=False))
print("\nSaved: experiment6_uncertainty_spread_by_area.csv")

print("\n\n===== EXPERIMENT 7: CALIBRATION =====")
print(calibration_df.to_string(index=False))
print("\nSaved: experiment7_calibration_by_area.csv")


AREA: DK1

  Model: Gaussian SDE-Net (alpha=2.0)
    horizon q=1, seed=0 ...
    horizon q=1, seed=1 ...
    horizon q=1, seed=2 ...
    horizon q=1, seed=3 ...
    horizon q=1, seed=4 ...
    horizon q=2, seed=0 ...
    horizon q=2, seed=1 ...
    horizon q=2, seed=2 ...
    horizon q=2, seed=3 ...
    horizon q=2, seed=4 ...
    horizon q=4, seed=0 ...
    horizon q=4, seed=1 ...
    horizon q=4, seed=2 ...
    horizon q=4, seed=3 ...
    horizon q=4, seed=4 ...

  Model: LévySDE-Net (alpha=1.5)
    horizon q=1, seed=0 ...
    horizon q=1, seed=1 ...
    horizon q=1, seed=2 ...
    horizon q=1, seed=3 ...
    horizon q=1, seed=4 ...
    horizon q=2, seed=0 ...
    horizon q=2, seed=1 ...
    horizon q=2, seed=2 ...
    horizon q=2, seed=3 ...
    horizon q=2, seed=4 ...
    horizon q=4, seed=0 ...
    horizon q=4, seed=1 ...
    horizon q=4, seed=2 ...
    horizon q=4, seed=3 ...
    horizon q=4, seed=4 ...


===== EXPERIMENT 6: PREDICTIVE UNCERTAINTY =====
Area            Model  Av